In [ ]:
import sys
import os

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'parameters'))
sys.path.insert(0, os.getcwd())

from analysis_utils import (
    load_all_runs, load_top_n_runs,
    plot_2d_histograms, plot_hp_sensitivity,
    print_hp_sensitivity_table, print_robustness_table,
    get_optimizer_colors,
    TASK_CONFIGS,
)

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

BACKEND = "local"
RESULTS_DIR = os.path.join('..', '..', 'results')

# Optional: filter to a subset of optimizers (None = all)
OPTIMIZERS_TO_PLOT = None  # e.g. ['adam', 'sgd_learn_diag_curv']


def load_hp_data(task_key):
    """Load all runs and top-N runs for HP analysis."""
    cfg = TASK_CONFIGS[task_key]
    optimizers = OPTIMIZERS_TO_PLOT or cfg['optimizers']
    colors = get_optimizer_colors(optimizers)

    all_runs = load_all_runs(
        backend=BACKEND, optimizers=optimizers,
        task_tag=cfg['task_tag'], results_dir=RESULTS_DIR,
        iteration=cfg['iteration'],
    )
    top_n_runs = load_top_n_runs(
        backend=BACKEND, optimizers=optimizers, n=50,
        task_tag=cfg['task_tag'], results_dir=RESULTS_DIR,
        metric_key=cfg['metric_key'], direction=cfg['direction'],
        sort_metric=cfg['sort_metric'], sort_order=cfg['sort_order'],
        iteration=cfg['iteration'],
    )
    return cfg, optimizers, all_runs, top_n_runs, colors


def show_hp_sensitivity(cfg, optimizers, all_runs, top_n_runs, colors):
    """Plot all HP sensitivity analysis for a task."""
    name = cfg['display_name']
    itr = cfg['iteration']

    # Sweep distribution
    fig = plot_2d_histograms(
        top_n_runs, backend=BACKEND,
        x_metric=cfg['hist_x'], y_metric=cfg['hist_y'],
        optimizers=optimizers,
    )
    plt.show()

    # HP sensitivity scatter grids
    for opt in optimizers:
        runs = all_runs.get(opt, [])
        if not runs:
            continue
        kwargs = dict(metric_key=cfg['metric_key'], direction=cfg['direction'])
        threshold = cfg.get('convergence_threshold')
        if threshold is not None:
            kwargs['convergence_threshold'] = threshold
        fig, axes = plot_hp_sensitivity(runs, opt, **kwargs)
        plt.show()

    # HP sensitivity tables
    for opt in optimizers:
        runs = all_runs.get(opt, [])
        if len(runs) >= 10:
            print_hp_sensitivity_table(runs, metric_key=cfg['metric_key'],
                                       title=f'HP Sensitivity — {opt}')

    # Robustness
    print_robustness_table(all_runs, metric_key=cfg['metric_key'],
                           direction=cfg['direction'],
                           title=f'{name} (itr {itr}) — Robustness')

# HP Sensitivity Analysis

Sweep distributions, HP sensitivity scatter grids, Spearman correlation tables, and robustness metrics.
Set `OPTIMIZERS_TO_PLOT` above to filter to a subset.

## MNIST MLP

In [ ]:
show_hp_sensitivity(*load_hp_data("mnist_mlp"))

## CIFAR-10 ResNet-18

In [ ]:
show_hp_sensitivity(*load_hp_data("cifar10_resnet18"))

## Shakespeare MiniGPT

In [ ]:
show_hp_sensitivity(*load_hp_data("shakespeare_minigpt"))

## Regression

In [ ]:
show_hp_sensitivity(*load_hp_data("regression"))